# Alur Kerja Pemodelan Bahaya Banjir HydroMT-SFINCS
Kernel: **HydroMT-SFINCS** (`env-sfincs`)

Notebook ini menjalankan secara otomatis **alur kerja pemodelan bahaya banjir hidrodinamika 2D HydroMT-SFINCS** untuk Provinsi Sumatera Barat:
1. **Kisi Komputasi 2D 80 m (tersedia opsi 40 m)** yang dipadukan dengan tabel **subkisi FABDEM 10 m** (komputasi sekitar 13 kali lebih cepat dengan ketelitian saluran 10 m).
2. **Hujan Langsung pada Kisi (*Direct Rainfall-on-Grid*)** menggunakan hidrograf hujan rancangan (`RP2` hingga `RP100`) pada seluruh klaster DAS aktif.
3. **Slot Integrasi Data BWS Modular** untuk penambahan hasil survei sungai, tanggul pengendali banjir, serta pos duga air AWLR.
4. **Eksekusi otomatis mesin simulasi SFINCS** (menggunakan berkas biner lokal Windows `sfincs.exe` atau cadangan Docker).
5. **Penurunan skala kedalaman banjir ke resolusi 10 m**, perhitungan **Indeks Bahaya Banjir (*Fuzzy Large*, 0 hingga 1)**, serta **klasifikasi bahaya banjir mengacu Peraturan Kepala BNPB Nomor 02 Tahun 2012**.

---
## PANEL KENDALI KONFIGURASI PENGGUNA (Ubah Parameter Hanya pada Bagian Ini)

> **PEDOMAN PELAKSANAAN BAGI PENGGUNA DAN ANGGOTA TIM:**  
> Anda **cukup menyesuaikan variabel pada sel di bawah ini**. Seluruh tahapan alur kerja berikutnya (Pembangunan Kisi Dasar, Pemaksaan Hujan, Simulasi SFINCS, Penurunan Skala 10 m, Klasifikasi BNPB, dan Visualisasi) akan menyesuaikan secara otomatis.
>
> ### Contoh Skenario Penggunaan Umum:
>
> * **Skenario A: Memproses Satu Klaster DAS dengan Satu Kala Ulang (contoh: DAS 102, RP100)**
>   ```python
>   SELECTED_CLUSTERS = ["DAS_102"]
>   SELECTED_RPS = ["rp100"]
>   ```
> * **Skenario B: Memproses Satu Klaster DAS untuk Seluruh Kala Ulang (contoh: DAS 101, RP2 hingga RP100)**
>   ```python
>   SELECTED_CLUSTERS = ["DAS_101"]
>   SELECTED_RPS = ["rp2", "rp5", "rp10", "rp25", "rp50", "rp100"]
>   ```
> * **Skenario C: Pemrosesan Massal Beberapa Klaster DAS untuk Kala Ulang 100 Tahun (RP100)**
>   ```python
>   SELECTED_CLUSTERS = ["DAS_101", "DAS_102"]
>   SELECTED_RPS = ["rp100"]
>   ```

In [ ]:
# ==============================================================================
# 1. TENTUKAN KLASTER DAS SASARAN DAN KALA ULANG DI SINI
# ==============================================================================

# Kode klaster DAS sasaran (mendukung berbagai format penamaan wilayah studi)
# Contoh: ["DAS_102"], ["DAS201"], ["305"], ["407"], ["Citarum_Hilir"]
SELECTED_CLUSTERS = ["DAS_101"]

# Pilih kala ulang: "rp2", "rp5", "rp10", "rp25", "rp50", "rp100"
SELECTED_RPS = ["rp100"]

# ==============================================================================
# 2. KONFIGURASI BATAS WILAYAH DAN DATA OPSIONAL BWS
# ==============================================================================
# Jalur berkas poligon batas klaster DAS (GeoPackage)
DAS_CLUSTERS_GPKG = "data/das_clusters/sumbar_das_cluster.gpkg"

# Data infrastruktur dan survei BWS bersifat opsional (biarkan None untuk pemodelan dasar)
BWS_LEVEES_GPKG = None        # contoh: "data_raw/bws/tanggul_sungai.gpkg"
BWS_GAUGES_GPKG = None        # contoh: "data_raw/bws/pos_awlr.gpkg"
BWS_INFLOW_HYDROGRAPHS = None # contoh: "data_raw/bws/debit_inflow.csv"

# ==============================================================================
# 3. DIREKTORI KELUARAN DAN BERKAS KONFIGURASI
# ==============================================================================
OUTPUT_MODELS_DIR = "models_sfincs_standalone"
OUTPUT_MAPS_DIR = "outputs"
CONFIG_FILE = "configs/sfincs_standalone_build_80m_fast.yml"  # Kisi fluks 80 m / subkisi 10 m baku (gunakan sfincs_standalone_build.yml untuk 40 m)
DATA_CATALOG_FILE = "data_catalog_sfincs.yml"


# ==============================================================================
# 4. PARAMETER HIDRODINAMIKA DAN PASCASIMULASI STANDAR NASIONAL v2.0 (80 m V3)
# ==============================================================================
STORM_HOURS = 24            # Durasi kejadian hujan rancangan (jam)
RECESSION_HOURS = None      # Otomatis berdasarkan luas DAS (24 jam <500 km2, 36 jam 500 hingga 1500 km2, 48 jam >1500 km2)
INFILTRATION_MODE = 'cnb'   # 'cnb' (USDA-NRCS smax/seff/ks standar v2.0) atau 'qinf' (infiltrasi konstan cadangan)
COASTAL_MHHW_M = 0.80       # Tinggi muka air pasang tertinggi rata-rata pesisir MHHW (m) pada sel msk=2 (0.0 menonaktifkan)
APPLY_ARF = False           # Opsi Faktor Reduksi Luas (ARF) USWB/WMO untuk DAS luas (>100 km2)
CLASSIFICATION_METHOD = 'depth'  # 'depth' (resmi Perka BNPB No. 02/2012) atau 'fuzzy'
COSMETIC = False            # False mempertahankan kedalaman genangan hidrostatis fisik
ISLAND_MMU_HA = 0.09        # Ambang minimum bercak terhubung 4 arah 30 m x 30 m (0,09 ha = 1 piksel FABDEM)
MIN_HOLE_HA = 0.09          # Ambang pengisian lubang mikro terhubung 4 arah 30 m x 30 m (0,09 ha)
HOLE_MMU_HA = 2.0           # Batas luas pengisian rongga tertutup hidrostatis (2,0 ha)
REPROJ_METHOD = 'bilinear'  # Interpolasi kontinu bilinear untuk elevasi muka air
BRIDGE_RADIUS = 6           # Radius penutupan morfologi melingkar 6 piksel (60 m) melintasi tanggul jalan
H_CONNECT = 0.10            # Ambang inti keterhubungan hidraulis (0,10 m)
FREEBOARD_M = 0.35          # Margin ketidakpastian vertikal FABDEM +0,35 m (< 0,50 m MAE, Hawker et al., 2022)
MAX_EXT_DIST_M = 120.0      # Jarak pemulihan tepi terhubung hidrologis 120 m (1,5 x dx)
MAX_EXT_SLOPE_DEG = 5.0     # Batas kemiringan lereng dataran banjir landai (< 5 derajat) untuk margin
SMOOTH_SIGMA = 1.2          # Parameter Gaussian penghalusan batas genangan (1,2 piksel)
MAX_SLOPE_DEG = 8.0         # Ambang penyaringan derau lereng curam (8,0 derajat; sungai RBI selalu dilindungi)

print(f"🎯 Active Target Clusters : {SELECTED_CLUSTERS}")
print(f"🎯 Active Return Periods  : {SELECTED_RPS}")
print(f"🗺️ Boundary Layer Source  : {DAS_CLUSTERS_GPKG}")
print(f"📂 Models Directory       : {OUTPUT_MODELS_DIR}")
print(f"🗺️ Hazard Maps Directory   : {OUTPUT_MAPS_DIR}")
print(f"⏱️ Simulation Timeline     : {STORM_HOURS}h storm + {'AUTO' if RECESSION_HOURS is None else f'{RECESSION_HOURS}h'} recession")
print(f"🌧️ Infiltration / Tide    : {INFILTRATION_MODE.upper()} | Coastal MHHW = +{COASTAL_MHHW_M:.2f}m | ARF = {APPLY_ARF}")
print(f"🏷️ Classification Method  : {CLASSIFICATION_METHOD.upper()} (Perka BNPB No. 02/2012)")
print(f"📐 Minimum Mapping Unit   : {HOLE_MMU_HA:.2f} ha hole / {ISLAND_MMU_HA:.2f} ha (30x30m) 4-conn floor")
print(f"🌊 Natural Extent (Opt B) : +{FREEBOARD_M:.2f}m FABDEM margin (<= {MAX_EXT_DIST_M:.0f}m, slope < {MAX_EXT_SLOPE_DEG}°)")


---
## 1. Verifikasi Lingkungan Kerja dan Data

Menginisialisasi lingkungan Python, menetapkan jalur proyeksi PROJ/GDAL secara dinamis, serta memuat seluruh fungsi modular dari `scripts/sfincs_standalone_pipeline.py`.

In [ ]:
# Menetapkan direktori kerja secara otomatis ke folder utama proyek
from pathlib import Path
import os, sys
project_root = Path.cwd()
while not (project_root / 'data').exists() and project_root.parent != project_root:
    project_root = project_root.parent
os.chdir(project_root)
if str(project_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(project_root / 'scripts'))

import sys
import os
from pathlib import Path
import pyproj

# Menambahkan direktori scripts lokal ke sys.path
if "scripts" not in sys.path:
    sys.path.insert(0, "scripts")

# Menetapkan jalur PROJ dan GDAL secara dinamis sesuai lingkungan Python aktif
site_packages = Path(sys.prefix) / 'Lib' / 'site-packages'
proj_dir = (site_packages / 'rasterio' / 'proj_data').resolve()
gdal_dir = (site_packages / 'rasterio' / 'gdal_data').resolve()
if proj_dir.exists():
    os.environ['PROJ_DATA'] = str(proj_dir)
    os.environ['PROJ_LIB'] = str(proj_dir)
    pyproj.datadir.set_data_dir(str(proj_dir))
if gdal_dir.exists():
    os.environ['GDAL_DATA'] = str(gdal_dir)

import sfincs_standalone_pipeline
import importlib
importlib.reload(sfincs_standalone_pipeline)
from sfincs_standalone_pipeline import (
    build_sfincs_standalone,
    prepare_sfincs_direct_rain_forcing,
    run_sfincs_model,
    postprocess_sfincs_hazard,
    create_sfincs_flood_animation,
)
from qa_checks import run_all_qa
from validate_against_survey import validate_model_against_survey

print("Environment initialized and Standalone SFINCS pipeline loaded successfully!")


---
## 2. Tahap 1: Pembangunan Kisi Dasar SFINCS (Kisi Baku 80 m / Opsi 40 m + Subkisi 10 m)

Membangun domain komputasi 2D terbatasi DAS, mengekstraksi tabel subkisi 10 m secara langsung dari FABDEM 30 m asli, melakukan pembakaran alur sungai RBI yang diperkaya poligon lebar `rivers_ar.gpkg`, serta memetakan kekasaran permukaan dari Penutup Lahan RBI BIG.

In [ ]:
for das_id in SELECTED_CLUSTERS:
    print(f"\n==================== [{das_id}] BUILDING BASE SFINCS MESH ====================")
    build_sfincs_standalone(
        das_id=das_id,
        output_dir=OUTPUT_MODELS_DIR,
        config_fn=CONFIG_FILE,
        data_catalog_fn=DATA_CATALOG_FILE,
        das_clusters_gpkg=DAS_CLUSTERS_GPKG,
        bws_structures_gpkg=BWS_LEVEES_GPKG,
        bws_gauges_gpkg=BWS_GAUGES_GPKG,
        force_overwrite=True,
    )


---
## 3. Tahap 2: Pemaksaan Hujan Langsung, Infiltrasi CNB, dan Pasang Surut MHHW

Menerapkan array hujan rancangan per jam berformat NetCDF (`rainfall_<rp>.nc`) beserta periode resesi nol hujan dinamis, skema infiltrasi Bilangan Kurva `CNB`, serta tinggi muka air pasang surut MHHW pada seluruh sel komputasi 2D.

In [ ]:
for das_id in SELECTED_CLUSTERS:
    print(f"\n==================== [{das_id}] PREPARING FORCINGS FOR {SELECTED_RPS} ====================")
    prepare_sfincs_direct_rain_forcing(
        das_id=das_id,
        return_periods=SELECTED_RPS,
        base_model_dir=OUTPUT_MODELS_DIR,
        data_catalog_fn=DATA_CATALOG_FILE,
        bws_inflow_hydrographs=BWS_INFLOW_HYDROGRAPHS,
        storm_hours=STORM_HOURS,
        recession_hours=RECESSION_HOURS,
        infiltration_mode=INFILTRATION_MODE,
        coastal_mhhw_m=COASTAL_MHHW_M,
        apply_arf=APPLY_ARF,
    )


---
## 4. Tahap 3: Eksekusi Mesin Simulasi Hidrodinamika SFINCS

Menjalankan penyelesai hidrodinamika 2D (mendeteksi otomatis berkas biner lokal Windows `bin/sfincs.exe` atau cadangan kontainer Docker `deltares/sfincs-cpu`).

In [ ]:
for das_id in SELECTED_CLUSTERS:
    for rp in SELECTED_RPS:
        print(f"\n==================== [{das_id}][{rp}] EXECUTING HYDRODYNAMIC SIMULATION ====================")
        run_sfincs_model(
            das_id=das_id,
            rp=rp,
            base_model_dir=OUTPUT_MODELS_DIR,
        )


---
## 5. Tahap 4: Penurunan Skala Kedalaman 10 m, Pengolahan Pascasimulasi Opsi B, Indeks Bahaya Banjir, dan Klasifikasi BNPB

Menurunkan skala elevasi muka air SFINCS ke topografi FABDEM resolusi 10 m, menerapkan **Margin Dataran Banjir Alami Terhubung Hidrostatis 80 m (`+0,35 m` margin ketidakpastian vertikal FABDEM yang dibatasi ketat pada dataran landai `< 5 derajat` dalam jarak `<= 120 m`)**, **Penjembatanan Hidromorfologi (`60 m`)**, **Penghalusan Batas ($\sigma = 1,2$)**, serta **pembersihan bercak dan lubang terhubung 4 arah `30 m x 30 m` (`0,09 ha`)** mengacu **Peraturan BIG Nomor 18 Tahun 2021, SNI 6502.3:2010, dan Peraturan Kepala BNPB Nomor 02 Tahun 2012**. Tahap ini juga menghitung **Indeks Bahaya Banjir (0 hingga 1)** menggunakan fungsi **Fuzzy Large** (Titik Tengah: `1,125`, Sebaran: `1,75`), mengklasifikasikan kelas bahaya sesuai **Peraturan Kepala BNPB Nomor 02 Tahun 2012**, serta menjalankan audit QA subkisi dan validasi survei QField.


In [ ]:
for das_id in SELECTED_CLUSTERS:
    for rp in SELECTED_RPS:
        print(f"\n==================== [{das_id}][{rp}] POST-PROCESSING & BNPB CLASSIFICATION ====================")
        postprocess_sfincs_hazard(
            das_id=das_id,
            rp=rp,
            base_model_dir=OUTPUT_MODELS_DIR,
            output_maps_dir=OUTPUT_MAPS_DIR,
            das_clusters_gpkg=DAS_CLUSTERS_GPKG,
            hmin=0.05,
            reproj_method=REPROJ_METHOD,
            island_mmu_ha=ISLAND_MMU_HA,
            hole_mmu_ha=HOLE_MMU_HA,
            min_hole_ha=MIN_HOLE_HA,
            bridge_radius=BRIDGE_RADIUS,
            smooth_sigma=SMOOTH_SIGMA,
            h_connect=H_CONNECT,
            freeboard_m=FREEBOARD_M,
            max_ext_dist_m=MAX_EXT_DIST_M,
            max_ext_slope_deg=MAX_EXT_SLOPE_DEG,
            classification_method=CLASSIFICATION_METHOD,
            cosmetic=COSMETIC,
            max_slope_deg=MAX_SLOPE_DEG,
        )
        # Pemeriksaan kendali mutu (qa_report.json) dan validasi lapangan QField (survey_validation.json)
        # dijalankan secara otomatis di dalam fungsi postprocess_sfincs_hazard().


---
## 6. Tahap 5: Visualisasi Peta Hasil Pemodelan

Menampilkan peta raster kedalaman genangan 10 m, Indeks Bahaya Banjir kontinu (*Fuzzy Large*, 0 hingga 1), serta peta klasifikasi bahaya banjir mengacu Peraturan Kepala BNPB Nomor 02 Tahun 2012 untuk verifikasi visual.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap, BoundaryNorm
import rasterio
import numpy as np
from pathlib import Path

# Select which completed run to inspect (defaults to first in list)
PLOT_DAS = SELECTED_CLUSTERS[0]
PLOT_RP = SELECTED_RPS[0]

das_tag = "DAS" + PLOT_DAS[4:] if PLOT_DAS.upper().startswith("DAS_") else (PLOT_DAS.upper() if PLOT_DAS.upper().startswith("DAS") else f"DAS{PLOT_DAS}")
rp_tag = f"RP{PLOT_RP[2:]}" if PLOT_RP.lower().startswith("rp") else f"RP{PLOT_RP}"
prefix = f"{das_tag}_{rp_tag}"
plot_dir = Path(OUTPUT_MAPS_DIR) / PLOT_DAS / PLOT_RP
depth_path = plot_dir / f"{prefix}_Flood_Depth.tif"
if not depth_path.exists():
    depth_path = plot_dir / "flood_depth_10m.tif"
fhi_path = plot_dir / f"{prefix}_FloodHazard_Index.tif"
if not fhi_path.exists():
    fhi_path = plot_dir / "flood_hazard_index.tif"
hazard_path = plot_dir / f"{prefix}_FloodHazard_Classes.tif"
if not hazard_path.exists():
    hazard_path = plot_dir / "hazard_class_perka2012.tif"

if depth_path.exists():
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 7))
    
    # 1. Flood Inundation Depth Map (10m FABDEM Downscaled)
    with rasterio.open(depth_path) as src_d:
        depth_data = src_d.read(1)
        depth_masked = np.where(depth_data < 0.05, np.nan, depth_data)
        # SFINCS raster arrays have dy > 0 (row 0 is south); origin='lower' ensures North is at the top
        origin_mode = "lower" if src_d.transform.e > 0 else "upper"
        im1 = ax1.imshow(depth_masked, cmap="Blues", vmin=0, vmax=3.0, origin=origin_mode)
        plt.colorbar(im1, ax=ax1, label="Potensi Ketinggian Genangan Banjir (m)", fraction=0.046, pad=0.04)
        ax1.set_title(f"Peta Potensi Ketinggian Genangan Banjir {PLOT_DAS} [{PLOT_RP}]", fontsize=11, fontweight="bold")
        ax1.axis("off")
        
    # 2. Continuous Flood Hazard Index (Fuzzy Large: 0 to 1)
    # Cartographic continuous ramp: Rendah (#34A853) -> Sedang (#FBBC05) -> Tinggi (#EA4335)
    cmap_fhi = LinearSegmentedColormap.from_list("fhi_continuous", ["#34A853", "#FBBC05", "#EA4335"])
    if fhi_path.exists():
        with rasterio.open(fhi_path) as src_f:
            fhi_data = src_f.read(1)
            im2 = ax2.imshow(fhi_data, cmap=cmap_fhi, vmin=0.0, vmax=1.0, origin=origin_mode)
            plt.colorbar(im2, ax=ax2, label="Hazard Index (0 - 1)", fraction=0.046, pad=0.04)
            ax2.set_title(f"Indeks Bahaya Banjir {PLOT_DAS} [{PLOT_RP}]\n(Fuzzy Large, Midpoint: 1.125, Spread: 1.75)", fontsize=11, fontweight="bold")
            ax2.axis("off")
            
    # 3. Perka BNPB No. 2/2012 Hazard Classification Map (Low, Medium, High)
    if hazard_path.exists():
        with rasterio.open(hazard_path) as src_h:
            hazard_data = src_h.read(1)
            hazard_masked = np.where(hazard_data == 0, np.nan, hazard_data)
            cmap_bnpb = ListedColormap(["#34A853", "#FBBC05", "#EA4335"])  # Rendah (#34A853), Sedang (#FBBC05), Tinggi (#EA4335)
            norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5], cmap_bnpb.N)
            im3 = ax3.imshow(hazard_masked, cmap=cmap_bnpb, norm=norm, origin=origin_mode)
            cbar = plt.colorbar(im3, ax=ax3, ticks=[1, 2, 3], fraction=0.046, pad=0.04)
            cbar.ax.set_yticklabels(["Rendah (h < 0.76 m)", "Sedang (0.76 - 1.50 m)", "Tinggi (h > 1.50 m)"])
            ax3.set_title(f"Kelas Bahaya Banjir {PLOT_DAS} [{PLOT_RP}]\n(Mengacu Perka BNPB No. 2/2012)", fontsize=11, fontweight="bold")
            ax3.axis("off")

    # Add cartographic North arrow to each map panel
    for ax in (ax1, ax2, ax3):
        ax.annotate(
            "U\n▲",
            xy=(0.95, 0.92),
            xycoords="axes fraction",
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            bbox=dict(boxstyle="square,pad=0.2", facecolor="white", alpha=0.7, edgecolor="#cccccc"),
        )
            
    plt.suptitle(f"Peta Model Bahaya Banjir: {PLOT_DAS} ({PLOT_RP.upper()})\n", fontsize=15, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print(f"Output not found yet: {depth_path}. Please execute Steps 1-4 first.")


---
## 7. Tahap 6: Animasi Perambatan Gelombang Banjir per Jam

Mengekstraksi irisan muka air dinamis per jam dari `sfincs_map.nc`, menumpangsusunkan kedalaman genangan terhadap model elevasi medan, serta menyusun berkas animasi GIF beresolusi tinggi (`DASXXX_RPYYY_Flood_Animation.gif`).


In [ ]:
from IPython.display import Image, display

for das_id in SELECTED_CLUSTERS:
    for rp in SELECTED_RPS:
        print(f"\n==================== [{das_id}][{rp}] GENERATING FLOOD PROPAGATION ANIMATION ====================")
        anim_path = create_sfincs_flood_animation(
            das_id=das_id,
            rp=rp,
            base_model_dir=OUTPUT_MODELS_DIR,
            output_maps_dir=OUTPUT_MAPS_DIR,
            das_clusters_gpkg=DAS_CLUSTERS_GPKG,
            kecamatan_gpkg="data/boundary_kecamatan.gpkg",
            kabkot_gpkg="data/boundary_kabkot.gpkg",
            lake_gpkg="data/lake.gpkg",
            fps=2,
            hmin=0.05,
            vmax=3.0,
        )
        display(Image(filename=anim_path))
